
# TATN Official Pretrained Transfer Dispatch Notebook

目标：**只做调度**，尽量不在 notebook 里重写 TATN 的核心逻辑。

本 notebook 做三件事：

1. 拉取 `zhangcastle/TATN` repo。
2. 加载 repo 自带的官方 pretrained models：`models/pre-25.pt`, `models/pre-10.pt`, `models/pre-0.pt`, `models/pre-n10.pt`, `models/pre-n20.pt`。
3. 调用 repo 里的 `pseudo.py` 和 `train.py` 跑半监督 transfer，先检查官方模型迁移效果。

默认先跑一个最小 case：

```text
source = 10°C
target = 0°C
test_cycle = Cycle_4.mat
labeled_cycle = 当前非 test cycles 的最后一个 cycle
```

跑通以后，再扩大到全部 folds / targets。


## Step 1：挂载 Drive，拉取 repo

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys, time, json, glob

REPO_URL = 'https://github.com/zhangcastle/TATN.git'
WORK_DIR = '/content/TATN'

os.chdir('/content')
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
subprocess.run(['git', 'clone', REPO_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)

print('工作目录:', os.getcwd())
print('repo 文件:', sorted(os.listdir('.')))


## Step 2：依赖与路径配置

In [ ]:

!pip install scipy tqdm scikit-learn -q

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# ===================== 用户需要确认的路径 =====================
# 这里应该指向已经处理好的 normalized_data/Pan 根目录。
# 结构必须是：
#   NORMALIZED_PAN_ROOT/10/10Cycle_1.mat
#   NORMALIZED_PAN_ROOT/0/0Cycle_4.mat
#   ...
# 如果你已经在 repo 内有 normalized_data/Pan，就保持默认即可。
# 改这里
NORMALIZED_PAN_ROOT = '/content/drive/MyDrive/Research/mining review/TATN/normalized_data/Pan'

# 如果 normalized data 存在 Drive，可以改成类似下面路径：
# NORMALIZED_PAN_ROOT = '/content/drive/MyDrive/Research/mining review/TATN/normalized_data/Pan'

OFFICIAL_MODEL_DIR = os.path.join(WORK_DIR, 'models')
RESULT_ROOT = '/content/drive/MyDrive/Research/mining review/TATN/results/official_pretrained_transfer'
os.makedirs(RESULT_ROOT, exist_ok=True)

print('NORMALIZED_PAN_ROOT:', NORMALIZED_PAN_ROOT)
print('OFFICIAL_MODEL_DIR:', OFFICIAL_MODEL_DIR)
print('RESULT_ROOT:', RESULT_ROOT)

# 基本检查
for p in [OFFICIAL_MODEL_DIR]:
    assert os.path.isdir(p), f'目录不存在: {p}'

for m in ['pre-25.pt', 'pre-10.pt', 'pre-0.pt', 'pre-n10.pt', 'pre-n20.pt']:
    print(m, 'exists =', os.path.exists(os.path.join(OFFICIAL_MODEL_DIR, m)))


## Step 3：实验配置（默认单 case）

In [ ]:

# ===================== 实验设置 =====================
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# 官方模型 isolation test：先用官方 pretrained source model，不用我们自己训练的 model
USE_OFFICIAL_PRETRAINED = True

# 先跑单个 case，确认 pipeline 是否接近论文
DEBUG_SINGLE_CASE = True
SOURCE_TEMP = '10'
TARGET_TEMPS = ['0'] if DEBUG_SINGLE_CASE else ['25', '0', 'n10', 'n20']
DEBUG_TEST_CYCLES = ['Cycle_4.mat'] if DEBUG_SINGLE_CASE else None

# 尽量接近原 run.py 默认设置
EPOCHS = 2000
EVAL_INTERVAL = 200
BATCH_SIZE = 64
LR = 1e-3
LAMB1 = 1.0
LAMB2 = 1.0
LAMB3 = 1.0
CRITERION_REDUCTION = 'sum'
LABEL_RATIO = 0.5
SEED = 100

print('DEVICE:', DEVICE)
print('SOURCE_TEMP:', SOURCE_TEMP)
print('TARGET_TEMPS:', TARGET_TEMPS)
print('DEBUG_SINGLE_CASE:', DEBUG_SINGLE_CASE)
print('EPOCHS:', EPOCHS, 'BATCH_SIZE:', BATCH_SIZE, 'LR:', LR)
print('lambdas:', LAMB1, LAMB2, LAMB3, 'criterion:', CRITERION_REDUCTION)


## Step 4：导入 repo 里的 py 文件

In [ ]:

import importlib
import numpy as np
import scipy.io as sio
import torch.nn as nn
import torch.optim as optim

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# 只导入 repo 里的模块，不在 notebook 重写模型/训练逻辑
for m in ['models', 'mydata', 'train', 'utils', 'pseudo']:
    if m in sys.modules:
        importlib.reload(sys.modules[m])

import mydata
import models as tatn_models
import train as tatn_train
import pseudo as tatn_pseudo

ALL_CYCLES = mydata.Pan_all_set
STEPS = mydata.steps

print('ALL_CYCLES:', ALL_CYCLES)
print('STEPS:', STEPS)


## Step 5：轻量调度函数（只做文件准备 + 调用 pseudo.py/train.py）

In [ ]:

def source_model_path(source_temp):
    if USE_OFFICIAL_PRETRAINED:
        return os.path.join(OFFICIAL_MODEL_DIR, f'pre-{source_temp}.pt')
    raise ValueError('当前 notebook 只用于 official pretrained model isolation test')


def check_cycle_exists(root, temp, cycle):
    path = os.path.join(root, temp, f'{temp}{cycle}')
    if not os.path.exists(path):
        raise FileNotFoundError(f'缺少数据文件: {path}')
    return path


def copy_true_cycle_to_workdir(src_root, dst_root, temp, cycle, max_samples=None):
    """只做文件准备：把 target true-label cycle 写到 train.py 可读取的目录中。"""
    src = check_cycle_exists(src_root, temp, cycle)
    mat = sio.loadmat(src)
    cur = mat['current'].reshape(-1, 1)
    vol = mat['voltage'].reshape(-1, 1)
    tmp = mat['temp'].reshape(-1, 1)
    ah = mat['ah'].reshape(-1, 1)

    n = min(len(cur), len(vol), len(tmp), len(ah))
    n = n // STEPS * STEPS
    if max_samples is not None:
        n = min(n, max_samples // STEPS * STEPS)
    if n < STEPS:
        raise ValueError(f'{cycle} valid length < {STEPS}: n={n}')

    out_subdir = os.path.join(dst_root, temp)
    os.makedirs(out_subdir, exist_ok=True)
    dst = os.path.join(out_subdir, f'{temp}{cycle}')
    sio.savemat(dst, {
        'current': cur[:n],
        'voltage': vol[:n],
        'temp': tmp[:n],
        'ah': ah[:n],
    })
    return dst, n


def count_mat_samples(path):
    m = sio.loadmat(path)
    return {k: len(m[k]) for k in ['current', 'voltage', 'temp', 'ah'] if k in m}


def prepare_fold_data_official(source_model, target_temp, test_cycle, labeled_cycle, pseudo_base='/tmp/tatn_official_pseudo'):
    """
    半监督 target 数据准备：
    - test_cycle: true labels，仅评估，不加入 train_set
    - labeled_cycle: 前 50% true labels，加入 train_set
    - 其他 cycles: 调用 repo 的 pseudo.py 生成 pseudo labels
    """
    fold_id = test_cycle.replace('.mat', '')
    pseudo_dir = os.path.join(pseudo_base, f'{SOURCE_TEMP}to{target_temp}_test_{fold_id}')
    if os.path.exists(pseudo_dir):
        shutil.rmtree(pseudo_dir)
    os.makedirs(os.path.join(pseudo_dir, target_temp), exist_ok=True)

    train_set = []
    pseudo_files = []

    for cycle in ALL_CYCLES:
        if cycle == test_cycle:
            # true-label test file must exist in the same target_data_path for train.py evaluation
            dst, n = copy_true_cycle_to_workdir(NORMALIZED_PAN_ROOT, pseudo_dir, target_temp, cycle)
            print(f'[test only] {cycle}: n={n} -> {dst}')

        elif cycle == labeled_cycle:
            # only first LABEL_RATIO labels are used
            src = check_cycle_exists(NORMALIZED_PAN_ROOT, target_temp, cycle)
            m = sio.loadmat(src)
            n_total = len(m['current']) // STEPS * STEPS
            n_labeled = max(STEPS, int(n_total * LABEL_RATIO) // STEPS * STEPS)
            dst, n = copy_true_cycle_to_workdir(NORMALIZED_PAN_ROOT, pseudo_dir, target_temp, cycle, max_samples=n_labeled)
            train_set.append(cycle)
            print(f'[labeled 50%] {cycle}: n={n}/{n_total} -> {dst}')

        else:
            # call repo pseudo.py logic, fallback disabled
            saved = tatn_pseudo.main(
                temp=target_temp,
                model_path=source_model,
                file=cycle,
                save_dir=pseudo_dir,
                data_path=NORMALIZED_PAN_ROOT + '/',
                enable_fallback=False,
            )
            for fname in saved:
                fpath = os.path.join(pseudo_dir, target_temp, f'{target_temp}{fname}')
                try:
                    lengths = count_mat_samples(fpath)
                    if lengths and min(lengths.values()) >= STEPS and len(set(lengths.values())) == 1:
                        train_set.append(fname)
                        pseudo_files.append(fname)
                        print(f'[pseudo] keep {fname}: {lengths}')
                    else:
                        print(f'[pseudo] skip {fname}: invalid lengths {lengths}')
                except Exception as e:
                    print(f'[pseudo] skip {fname}: {e}')

    return pseudo_dir + '/', train_set, pseudo_files


def make_official_models_and_optimizers(device, lr):
    """镜像 run.py 的 model/optimizer 构造。"""
    d = torch.device(device)
    mdls = {
        'conv': tatn_models.conv().to(d),
        'lstm': tatn_models.lstm().to(d),
        'fc': tatn_models.fc().to(d),
        'regression': tatn_models.regression().to(d),
        'conv_s': tatn_models.conv().to(d),
        'lstm_s': tatn_models.lstm().to(d),
        'fc_s': tatn_models.fc().to(d),
        'regression_s': tatn_models.regression().to(d),
        'discriminator': tatn_models.Discriminator().to(d),
    }
    opts = {
        'conv': optim.Adam(mdls['conv'].parameters(), lr=lr),
        'lstm': optim.Adam(mdls['lstm'].parameters(), lr=lr),
        'fc': optim.Adam(mdls['fc'].parameters(), lr=lr),
        'regression': optim.Adam(mdls['regression'].parameters(), lr=lr),
        'discriminator': optim.Adam(mdls['discriminator'].parameters(), lr=lr),
    }
    return mdls, opts

print('调度函数准备完成')


## Step 6：运行单个 official-pretrained transfer case

In [ ]:

all_results = {}
model_path = source_model_path(SOURCE_TEMP)
assert os.path.exists(model_path), f'找不到官方 pretrained model: {model_path}'
print('Using official source model:', model_path)

for target_temp in TARGET_TEMPS:
    test_cycles = DEBUG_TEST_CYCLES if DEBUG_SINGLE_CASE else ALL_CYCLES
    all_results[target_temp] = {}

    for test_cycle in test_cycles:
        train_cycles = [c for c in ALL_CYCLES if c != test_cycle]
        labeled_cycle = train_cycles[-1]

        print('\n' + '='*80)
        print(f'Transfer: {SOURCE_TEMP}°C -> {target_temp}°C | test={test_cycle} | labeled={labeled_cycle}')
        print('='*80)

        pseudo_dir, target_train_set, pseudo_files = prepare_fold_data_official(
            source_model=model_path,
            target_temp=target_temp,
            test_cycle=test_cycle,
            labeled_cycle=labeled_cycle,
        )

        print('\nPrepared target training set:')
        print('  pseudo_dir:', pseudo_dir)
        print('  target_train_set:', target_train_set)
        print('  num pseudo files:', len(pseudo_files))
        print('  target_test_set:', [test_cycle])

        assert test_cycle not in target_train_set, 'ERROR: test cycle leaked into target_train_set'
        if len(target_train_set) == 0:
            print('WARNING: target_train_set empty; skip this fold')
            continue

        # reload repo modules before training
        for m in ['models', 'train', 'utils']:
            if m in sys.modules:
                importlib.reload(sys.modules[m])
        import models as tatn_models
        import train as tatn_train

        mdls, opts = make_official_models_and_optimizers(DEVICE, LR)
        criterion = nn.MSELoss(reduction=CRITERION_REDUCTION)

        run_tag = f'official_pre_{SOURCE_TEMP}to{target_temp}_{test_cycle.replace(".mat", "")}'
        t0 = time.time()
        min_mae, min_rmse, min_max = tatn_train.train(
            rundir=run_tag,
            source_temp=SOURCE_TEMP,
            target_temp=target_temp,
            source_data_path=NORMALIZED_PAN_ROOT + '/',
            source_train_set=ALL_CYCLES,
            source_test_set=[ALL_CYCLES[0]],
            target_data_path=pseudo_dir,
            target_train_set=target_train_set,
            target_test_set=[test_cycle],
            models=mdls,
            criterion=criterion,
            optimizers=opts,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            eval_interval=EVAL_INTERVAL,
            lamb1=LAMB1,
            lamb2=LAMB2,
            lamb3=LAMB3,
            seed=SEED,
            device_type=DEVICE,
            ifsave=True,
            load_model=True,
            model_path=model_path,
        )
        elapsed = (time.time() - t0) / 60

        result = {
            'mae': float(min_mae[0]) if min_mae else float('nan'),
            'rmse': float(min_rmse[0]) if min_rmse else float('nan'),
            'max': float(min_max[0]) if min_max else float('nan'),
            'elapsed_min': elapsed,
            'target_train_set': target_train_set,
            'pseudo_files': pseudo_files,
            'source_model': model_path,
        }
        all_results[target_temp][test_cycle.replace('.mat', '')] = result

        print('\nRESULT')
        print(f"MAE={result['mae']*100:.3f}%  RMSE={result['rmse']*100:.3f}%  MAX={result['max']*100:.3f}%  time={elapsed:.1f} min")

print('\nAll done')
print(json.dumps(all_results, indent=2))


## Step 7：保存结果

In [ ]:

out_json = os.path.join(RESULT_ROOT, f'official_pretrained_transfer_{SOURCE_TEMP}.json')
with open(out_json, 'w') as f:
    json.dump(all_results, f, indent=2)
print('saved:', out_json)

# 同步当前 run 目录到 Drive，方便之后看图和 errors/min_errors
run_dirs = sorted(glob.glob(os.path.join(WORK_DIR, 'run', 'official_pre_*')))
print('run dirs:', run_dirs[-5:])
for rd in run_dirs:
    dst = os.path.join(RESULT_ROOT, os.path.basename(rd))
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(rd, dst)
print('copied run dirs to:', RESULT_ROOT)



## 说明

这份 notebook 的原则：

- 不重新实现 TATN 模型。
- 不重新实现训练过程。
- 不重新设计 pseudo-label 算法。
- 只做文件组织和调度，核心操作调用 repo 中的：
  - `pseudo.py -> pseudo.main(...)`
  - `train.py -> train(...)`
  - `models.py -> conv/lstm/fc/regression/Discriminator`

如果这个 single-case 结果接近论文，再把 `DEBUG_SINGLE_CASE=False` 扩展到完整 Table IV。
